# RNN for Classifying Names

In this notebook we are building and training a basic character-level RNN to classify
words. A character-level RNN reads words as a series of characters -
outputting a prediction and "hidden state" at each step, feeding its
previous hidden state into each next time step. We take the final prediction
to be the output, i.e. which class the word belongs to.

### Preparing the Data

Download the data in folder `data/names` from GitHub.

Included in the ``data/names`` directory are 18 text files named as
``[Language].txt``. Each file contains a bunch of names, one name per
line, mostly romanized (but we still need to convert from Unicode to
ASCII).

We first get all the filenames:

In [2]:
import glob
filenames = glob.glob('data/names/*.txt')

print(filenames)

['data/names/Czech.txt', 'data/names/German.txt', 'data/names/Arabic.txt', 'data/names/Japanese.txt', 'data/names/Chinese.txt', 'data/names/Vietnamese.txt', 'data/names/Russian.txt', 'data/names/French.txt', 'data/names/Irish.txt', 'data/names/English.txt', 'data/names/Spanish.txt', 'data/names/Greek.txt', 'data/names/Italian.txt', 'data/names/Portuguese.txt', 'data/names/Scottish.txt', 'data/names/Dutch.txt', 'data/names/Korean.txt', 'data/names/Polish.txt']


And save each language as a category:

In [3]:
import os
all_categories = []


for filename in filenames:
    language = os.path.splitext(os.path.basename(filename))[0]
    all_categories.append(language)

print(all_categories)

['Czech', 'German', 'Arabic', 'Japanese', 'Chinese', 'Vietnamese', 'Russian', 'French', 'Irish', 'English', 'Spanish', 'Greek', 'Italian', 'Portuguese', 'Scottish', 'Dutch', 'Korean', 'Polish']


Next we load the data and put every name in a list together and its category (=label) in a second list:

In [4]:
X = []
y = []


for index, filename in enumerate(filenames):
    lines = open(filename, encoding='utf-8').read().strip().split('\n')
    category = all_categories[index]
    for line in lines:
        X.append(line)
        y.append(category)

n_categories = len(all_categories)
n_categories, len(X)

(18, 20074)

Let's check which characters are included in the names:

In [5]:
all_characters = set([c for name in X for c in name])
print(all_characters)
print(len(all_characters), "characters")

{'É', 'ç', 'è', 'ß', 'V', 'D', 'd', 'ä', 'ń', 't', 'Á', 'i', 'r', ',', 'y', '1', 's', 'ã', 'a', 'ò', 'é', 'N', 'T', 'ł', 'n', 'G', 'á', 'ü', 'f', 'Q', 'J', '\xa0', 'ú', 'c', 'ù', 'K', 'u', '-', 'k', 'í', 'X', 'q', 'l', 'x', 'ö', ' ', 'E', 'W', 'B', 'A', 'ì', 'R', 'g', "'", 'e', ':', 'à', 'ñ', 'O', 'Z', 'h', 'C', 'S', 'Ż', 'P', 'ê', 'U', 'L', 'ó', 'b', 'j', 'F', 'õ', '/', 'p', 'm', 'Y', 'M', 'ż', 'v', 'z', 'I', 'Ś', 'ą', 'o', 'w', 'H'}
87 characters


We see that the files contain many special characters that make our problem more difficult. To reduce the character count, we only allow ASCII symbols:

In [6]:
import string

# these is the vocabulary we will use
all_letters = string.ascii_letters
n_letters = len(all_letters)

print(f"Vocab is of size {n_letters} and contains:", all_letters)

Vocab is of size 52 and contains: abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ


In [7]:
import unicodedata

# this method converts anything into ascii
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
        and c in all_letters
    )

print(unicodeToAscii('Ślusàrski'))
print(unicodeToAscii('Frühling'))

Slusarski
Fruhling


In [8]:
# convert all letters to ascii
X = [unicodeToAscii(x) for x in X]

# print again all characters
all_characters = set([c for name in X for c in name])
print(all_characters)
print(len(all_characters), "characters")

{'W', 'B', 'P', 'n', 'U', 'L', 'T', 'b', 'G', 'j', 'F', 'A', 'f', 'V', 'Q', 'J', 'p', 'D', 'm', 'd', 'M', 'Y', 'c', 't', 'R', 'K', 'i', 'g', 'u', 'r', 'v', 'k', 'e', 'y', 'X', 'q', 'z', 's', 'l', 'O', 'I', 'a', 'x', 'Z', 'h', 'o', 'w', 'C', 'H', 'N', 'S', 'E'}
52 characters


We can see that we successfully reduced the number of characters and can now divide the data into train and test data:

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

print("Train data points:", len(X_train))

Train data points: 16059


Turning Names into Tensors
--------------------------

Now that we have all the names organized, we need to turn them into
Tensors to make any use of them.

To represent a single letter, we use a "one-hot vector" of size
``<1 x n_letters>``. A one-hot vector is filled with 0s except for a 1
at index of the current letter, e.g. ``"b" = <0 1 0 0 0 ...>``.

To make a word we join a bunch of those into a 2D matrix
``<line_length x 1 x n_letters>``.

That extra 1 dimension is because PyTorch assumes everything is in
batches - we're just using a batch size of 1 here.




In [10]:
import torch

def letterToTensor(letter):
    tensor = torch.zeros(1, n_letters)
    index = all_letters.find(letter)
    tensor[0][index] = 1
    return tensor

Know lets check how the encoding of one letter looks like:

In [11]:
print(letterToTensor('J'))

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])


We also need to convert the label into a number, which is just the index of the category:

In [12]:
def categoryToTensor(category):
    index = all_categories.index(category)
    return torch.tensor([index], dtype=torch.long)

categoryToTensor("Korean")

tensor([16])

Creating the RNN
====================

This RNN module has two linear layers. One calculates the next hidden state, the other one the output.

In [ ]:
import torch.nn as nn
import gc

gc.collect()

class RNN(nn.Module):
    def __init__(self, input_size, output_size):
        super(RNN, self).__init__()

        self.hidden_size = 128 # number of hidden layer size

        self.input2hidden = nn.Linear(input_size + self.hidden_size, self.hidden_size)
        self.input2output = nn.Linear(input_size + self.hidden_size, output_size)

    def forward(self, x, hidden):
        combined = torch.cat((x, hidden), 1) # input and hidden state are combined
        hidden = self.input2hidden(combined) # calculate next hidden state
        output = self.input2output(combined) # calculate output state
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, self.hidden_size)

To run a step of this network we need to pass an input (in our case, the
tensor for the current letter) and a previous hidden state (which we
initialize as zeros at first). We get back the output and a next hidden state (which we keep for the next
step).




In [21]:
rnn = RNN(n_letters, n_categories)

x = letterToTensor('A')
hidden = rnn.initHidden()

output, next_hidden = rnn(x, hidden)
print(torch.softmax(output, 1))

tensor([[0.0513, 0.0575, 0.0564, 0.0494, 0.0534, 0.0575, 0.0553, 0.0591, 0.0584,
         0.0520, 0.0616, 0.0589, 0.0508, 0.0616, 0.0522, 0.0612, 0.0502, 0.0534]],
       grad_fn=<SoftmaxBackward0>)


As you can see the output is a ``<1 x n_categories>`` Tensor, where
every item is the likelihood of that category (higher is more likely).




Task 1: Training the Network
--------------------

Finish the following training function to train the RNN on the training data set.

In [22]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

torch.manual_seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(rnn.parameters(), lr=0.005)

all_losses = []
current_loss = 0

for epoch in range(1, 10):
    print("Training epoch:", epoch)
    # iterate through all names in X_train
    # for every name:
        # init the hidden layer of the rnn
        # insert the name character by character into the rnn and compute the final output
        # note: you need to carry on the hidden state in every time step
        # define the loss on the last output of the rnn and the category (=label)
        # backpropagate the loss and take an optimizer step
    for i in range(len(X_train)):
        name = X_train[i]
        
        hidden = rnn.initHidden()
        for c in unicodeToAscii(name):
            x = letterToTensor(c)
            output, hidden = rnn(x, hidden)
            
        optimizer.zero_grad()
        loss = criterion(output, categoryToTensor(y_train[i]))
        loss.backward()
        optimizer.step()
        current_loss += loss.item()
    
    all_losses.append(current_loss)
    print(f"Loss in epoch {epoch} is {current_loss/len(X_train)}")
    current_loss = 0

Using device: cpu
Training epoch: 1
Loss in epoch 1 is 1.4274110386230237
Training epoch: 2
Loss in epoch 2 is 1.1179816645740273
Training epoch: 3
Loss in epoch 3 is 1.0124757576500538
Training epoch: 4
Loss in epoch 4 is 0.9546465499778077
Training epoch: 5
Loss in epoch 5 is 0.9155800072270286
Training epoch: 6
Loss in epoch 6 is 0.887841679147895
Training epoch: 7
Loss in epoch 7 is 0.8670549724468546
Training epoch: 8
Loss in epoch 8 is 0.8508160218083776
Training epoch: 9
Loss in epoch 9 is 0.8376323055112234


### Task 2: Evaluating the Results

Evaluate the accuarcy of the RNN on the test data.

In [30]:
def predict(input_name, k=1):
    results = []
    rnn.eval()
    hidden = rnn.initHidden()
    for c in unicodeToAscii(input_name):
        x = letterToTensor(c)
        output, hidden = rnn(x, hidden)
        output = torch.softmax(output, 1)
    topk_probs, topk_indices = output.topk(k, dim=1)
    results = [(all_categories[topk_indices[0][i].item()], topk_probs[0][i].item()) for i in range(k)]
    return results

correct = 0
with torch.no_grad():
    for i in range(len(X_test)):
        name = X_test[i]
        label = y_test[i]
        pred = predict(name, k=1)[0][0]
        if pred == label:
           correct += 1

acc = correct / len(X_test)
print(f"Test accuracy: {acc*100:.2f}%")

Test accuracy: 75.02%


### Task 3: Running on User Input

Write a function that takes an abritrary name as input and outputs the top 3 categories of the RNN for the input.


In [34]:
def predictTop3(input_name):
    return predict(input_name, k=3)

print(predictTop3("Dovesky"))
print(predictTop3("Satoshi"))
print(predictTop3("Schwartz"))
print(predictTop3("Wang"))

[('Russian', 0.9007420539855957), ('English', 0.0453464575111866), ('Czech', 0.03592938557267189)]
[('Japanese', 0.4441303610801697), ('Arabic', 0.1443643420934677), ('Russian', 0.0957544595003128)]
[('German', 0.5661832094192505), ('English', 0.2146303951740265), ('Italian', 0.048912886530160904)]
[('Chinese', 0.3801881968975067), ('English', 0.2581420838832855), ('Korean', 0.09155384451150894)]
